In [1]:
%load_ext autoreload
%autoreload 2

import sys
import json
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

sys.path.append("../src")

from config import ExperimentConfig, with_overrides
from main import run_experiment

In [2]:
# Cell 2: Run INT8 Quantization (CPU)
# We use a smaller batch size because CPU inference is slower and memory-heavy
IMAGENET_PATH = "/home/pf4636/imagenet2"

cfg_int8 = ExperimentConfig(
    imagenet_path=IMAGENET_PATH,
    device="cpu",      # Explicitly set CPU for clarity (though main.py enforces it)
    batch_size=32,     # Smaller batch size for CPU
    num_workers=4,
    input_quant_bits=8,
    model_precision="int8"
)

print("--- Running INT8 Quantization (PT2E Static) ---")
# This will:
# 1. Load Data
# 2. Calibrate the model (taking ~1-2 mins)
# 3. Evaluate on the full validation set
results_int8 = run_experiment(
    cfg_int8, 
    split="val", 
    save_results_flag=True
)

--- Running INT8 Quantization (PT2E Static) ---
EXPERIMENT CONFIGURATION
Model precision:       int8
Input quantization:    8-bit
Batch size:            32
Num workers:           4
Device:                cpu
ImageNet path:         /home/pf4636/imagenet2
Num eval batches:      All

Loading val data...
Dataloader ready: 1563 batches

Loading model...
Starting PT2E Static Quantization on cpu...
  Step 1: Exporting model graph...
  Step 2: Preparing observers (X86Inductor Config)...
  Step 3: Calibrating with representative data...


/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/torchao/quantization/pt2e/quantizer/x86_inductor_quantizer.py:1325: UserWarning: The input of maxpool2d is not quantized, skip annotate maxpool2d with config QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), weight=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.PerChannelMinMaxObserver'>, eps=0.000244140625){}, quant

AssertionError: Guard failed: x.size()[0] == 1

In [ ]:
# Cell 3: Compare Results (Thesis Visualization)
# This cell loads the saved JSONs from both notebooks and plots them.

def load_metrics(precision, device):
    path = Path(f"../results/resnet18_{precision}_{device}/metrics.json")
    if path.exists():
        with open(path) as f:
            return json.load(f)
    print(f"Warning: Could not find results for {precision} at {path}")
    return None

# Load data
res_fp32 = load_metrics("fp32", "cuda")
res_int8 = load_metrics("int8", "cpu")

if res_fp32 and res_int8:
    labels = ['FP32 (GPU)', 'INT8 (CPU)']
    accs = [res_fp32['top1_acc'], res_int8['top1_acc']]
    # Throughput is usually better for comparison than raw latency across diff hardware
    throughputs = [res_fp32['throughput_sps'], res_int8['throughput_sps']]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot Accuracy (Bars)
    bars = ax1.bar(x, accs, width, label='Top-1 Accuracy (%)', color=['#1f77b4', '#ff7f0e'])
    ax1.set_ylabel('Accuracy (%)')
    ax1.set_title('ResNet-18: Precision vs. Accuracy & Throughput')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.set_ylim(0, 100)

    # Plot Throughput (Line) - Dual Axis
    ax2 = ax1.twinx()
    ax2.plot(x, throughputs, color='green', marker='o', linewidth=2, label='Throughput')
    ax2.set_ylabel('Throughput (samples/sec)')
    
    # Add values on top of bars
    for i, v in enumerate(accs):
        ax1.text(i, v + 1, f"{v:.2f}%", ha='center', fontweight='bold')

    plt.show()
else:
    print("Run both experiments to generate the comparison plot!")